# Accessing Data

In [ ]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("/home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd /home/565/pv3484/aus_substation_electricity

!pwd

In [ ]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

In [ ]:
# info.head()

weather = pd.read_csv("/home/565/pv3484/aus_substation_electricity/data/BOM_NSW_weather_processed_v3/weather_holiday_block_means_with_codes.csv")

weather.head()

# Adding lat and lon into the info variable

In [ ]:
from geopy.geocoders import Nominatim
import pandas as pd
import time

# Initialize geocoder
geolocator = Nominatim(user_agent="sydney_demand_mapper")

def get_coords(place):
    """Return (lat, lon) for a suburb name, or (None, None) if not found."""
    try:
        loc = geolocator.geocode(f"{place}, New South Wales, Australia")
        if loc:
            return loc.latitude, loc.longitude
    except Exception as e:
        print(f"Geocoding failed for {place}: {e}")
    return None, None

# Apply geocoding to the 'Name' column
latitudes, longitudes = [], []
for suburb in info['Name']:
    lat, lon = get_coords(suburb)
    latitudes.append(lat)
    longitudes.append(lon)
    time.sleep(1)  # polite pause to avoid hitting API limits

info['latitude'] = latitudes
info['longitude'] = longitudes

In [ ]:
info.head(20)

In [ ]:
info.loc[:, ['Name', 'latitude', 'longitude']]

In [ ]:
# Saving Name, lat and lon information as comma-separated text file
info[['Name', 'latitude', 'longitude']].to_csv('name_lat_lon.txt', index=False)


## Trouble shooting Dee Why West

In [ ]:
missing = info[info["latitude"].isna() | info["longitude"].isna()]
missing["Name"].unique()
#Dee Why West doesn't exist as a suburb polygon, so will need to change the name to Dee Why

In [ ]:
dee_why_west_lat = -33.73441
dee_why_west_lon = 151.28278
#Found the lat/lon information online

In [ ]:
info.loc[info["Name"] == "Dee Why West", "latitude"] = dee_why_west_lat
info.loc[info["Name"] == "Dee Why West", "longitude"] = dee_why_west_lon
#inputting lat and lon from online into info

# Mapping

## Mapping BLAKE, CHATS, LIDC and GATES for relative ranks

In [ ]:
subset = info[info["Name"].isin(["Blakehurst", "Chatswood", "Lidcombe", "Gateshead"])]

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

# Filter the four substations
subset = info[info["energy_asset"].str.contains("BLAKE|CHATS|LIDC|GATES")].copy()

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(
    subset,
    geometry=gpd.points_from_xy(subset.longitude, subset.latitude),
    crs="EPSG:4326"
).to_crs("EPSG:3857")  # Web Mercator for basemap

# Plot
fig, ax = plt.subplots(figsize=(7, 9))
gdf.plot(ax=ax, color="red", markersize=80)

# Add labels using the Name column
for _, row in gdf.iterrows():
    ax.text(
        row.geometry.x + 200, 
        row.geometry.y + 200,
        row["Name"], 
        fontsize=10
    )

# Add basemap (stable provider)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

plt.title("Locations of Blakehurst, Chatswood, Lidcombe, Gateshead")
plt.tight_layout()

import pyproj
import numpy as np
from matplotlib.ticker import FixedLocator

# Transformer from Web Mercator → WGS84 lat/lon
to_lonlat = pyproj.Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

# Get current tick positions (in Web Mercator)
xticks_3857 = ax.get_xticks()
yticks_3857 = ax.get_yticks()

# Convert to lon/lat
lon_ticks, lat_ticks = to_lonlat.transform(xticks_3857, yticks_3857)

# Set tick positions explicitly
ax.xaxis.set_major_locator(FixedLocator(xticks_3857))
ax.yaxis.set_major_locator(FixedLocator(yticks_3857))

# Set tick labels
ax.set_xticklabels([f"{lon:.3f}" for lon in lon_ticks])
ax.set_yticklabels([f"{lat:.3f}" for lat in lat_ticks])
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

# Axis labels
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")


plt.show()


## Mapping all

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

# 1. Convert ALL substations to a GeoDataFrame in EPSG:4326
gdf = gpd.GeoDataFrame(
    info,
    geometry=gpd.points_from_xy(info.longitude, info.latitude),
    crs="EPSG:4326"
)

# 2. Plot directly in geographic coordinates
fig, ax = plt.subplots(figsize=(10, 12))
gdf.plot(ax=ax, color="red", markersize=20, alpha=0.8)

# Optional: label each point
for _, row in gdf.iterrows():
    ax.text(
        row.geometry.x + 0.01,
        row.geometry.y + 0.01,
        row["Name"],
        fontsize=6
    )

# 3. Add basemap — Contextily will reproject automatically
ctx.add_basemap(ax, crs=gdf.crs, source=ctx.providers.CartoDB.Positron)

# 4. Titles and labels
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.title("All Substation Locations")
plt.tight_layout()
plt.show()


## Creating a list of substations closest to furthest from coastline

In [ ]:
#downloading natural earth coastline at 10m resolution

import requests, zipfile, io, geopandas as gpd

url = "https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_coastline.zip"
r = requests.get(url)

z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall("natural_earth_coastline")

coast = gpd.read_file("/home/565/pv3484/aus_substation_electricity/data/natural_earth_coastline/ne_10m_coastline.shp")


In [ ]:
#Australia segment of global coastline

# Rough bounding box for Australia
minx, miny = 110, -45
maxx, maxy = 155, -10

coast_aus = coast.cx[minx:maxx, miny:maxy]


In [ ]:
#coverting datasets for distance calculations

info_gdf = gpd.GeoDataFrame(
    info,
    geometry=gpd.points_from_xy(info.longitude, info.latitude),
    crs="EPSG:4326"
).to_crs("EPSG:3857")

coast_3857 = coast_aus.to_crs("EPSG:3857")


In [ ]:
#computing distance in meters
info_gdf["dist_to_coast_m"] = info_gdf.geometry.apply(
    lambda p: coast_3857.distance(p).min()
)

#convert to km
info_gdf["dist_to_coast_km"] = info_gdf["dist_to_coast_m"] / 1000


In [ ]:
#sorting closest to furthest and residential fraction
ranked = info_gdf.sort_values("dist_to_coast_km")[
    ["energy_asset", "Name", "Residential", "dist_to_coast_km"]
]


In [ ]:
print(ranked.to_string(index=False))


In [ ]:
subset = info_gdf[
    info_gdf["Name"].isin(["Blakehurst", "Chatswood", "Gateshead", "Lidcombe"])
][["Name", "Residential", "dist_to_coast_km"]]

print(subset.to_string(index=False))


# Weather station to Substation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KDTree
import geopandas as gpd
from shapely.geometry import Point

# -----------------------------
# 1. Load your datasets
# -----------------------------

weather_path = "/home/565/pv3484/aus_substation_electricity/data/raw_data/All_weatherstations_information.xlsx"
substation_path = "/home/565/pv3484/aus_substation_electricity/pia_notebooks/NSW_data/relative_ranking/name_lat_lon.txt"

# Load substations (comma-separated)
subs = pd.read_csv(substation_path, sep=",", engine="python")
subs = subs.rename(columns=str.lower)

subs = subs.rename(columns={
    "latitude": "lat",
    "longitude": "lon"
})

weather = pd.read_excel(weather_path)
weather = weather.rename(columns=str.lower)

# FIX: rename to the names your code expects
weather = weather.rename(columns={
    "latitude": "lat",
    "longitude": "lon"
})

gdf_weather = gpd.GeoDataFrame(
    weather,
    geometry=gpd.points_from_xy(weather.lon, weather.lat),
    crs="EPSG:4326"
)

# -----------------------------
# 2. Convert to GeoDataFrames
# -----------------------------

gdf_weather = gpd.GeoDataFrame(
    weather,
    geometry=gpd.points_from_xy(weather.lon, weather.lat),
    crs="EPSG:4326"
)

gdf_subs = gpd.GeoDataFrame(
    subs,
    geometry=gpd.points_from_xy(subs.lon, subs.lat),
    crs="EPSG:4326"
)

# Convert to metric CRS for distance calculations (Australia Albers)
gdf_weather = gdf_weather.to_crs("EPSG:3577")
gdf_subs = gdf_subs.to_crs("EPSG:3577")

# -----------------------------
# 3. Build KD-tree on weather stations
# -----------------------------

weather_coords = np.vstack([gdf_weather.geometry.x, gdf_weather.geometry.y]).T
tree = KDTree(weather_coords)

# -----------------------------
# 4. Query nearest weather station for each substation
# -----------------------------

sub_coords = np.vstack([gdf_subs.geometry.x, gdf_subs.geometry.y]).T
distances, indices = tree.query(sub_coords, k=1)

gdf_subs["nearest_station_id"] = gdf_weather.iloc[indices.flatten()].station_number.values
gdf_subs["distance_km"] = distances.flatten() / 1000

# -----------------------------
# 5. Save mapping table
# -----------------------------

output_path = "/home/565/pv3484/aus_substation_electricity/pia_notebooks/NSW_data/relative_ranking/substation_to_weather_mapping.csv"

gdf_subs.drop(columns="geometry").to_csv(output_path, index=False)


print("Saved:", output_path)


In [ ]:
print(subs.columns)


In [ ]:
import pandas as pd

mapping_path = "/home/565/pv3484/aus_substation_electricity/pia_notebooks/NSW_data/relative_ranking/substation_to_weather_mapping.csv"

df = pd.read_csv(mapping_path)

unique_ids = df["nearest_station_id"].drop_duplicates()

print(unique_ids.to_list())


In [ ]:
unique_ids.to_csv(
    "/home/565/pv3484/aus_substation_electricity/pia_notebooks/NSW_data/relative_ranking/unique_weather_station_ids.txt",
    index=False,
    header=False
)


# Mapping all substations with residential fractions over 0.75

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
import pyproj
import numpy as np
from matplotlib.ticker import FixedLocator

# -----------------------------
# Filter data
# -----------------------------
high_res = info[info["Residential"] >= 0.75].copy()

# -----------------------------
# Convert to GeoDataFrame
# -----------------------------
gdf = gpd.GeoDataFrame(
    high_res,
    geometry=gpd.points_from_xy(high_res.longitude, high_res.latitude),
    crs="EPSG:4326"
).to_crs("EPSG:3857")

# -----------------------------
# Plot (auto-zoom)
# -----------------------------
fig, ax = plt.subplots(figsize=(8, 10))
gdf.plot(ax=ax, color="red", markersize=40)

# -----------------------------
# Labels (600 m to the right)
# -----------------------------
x_offset = 600  # metres

for _, row in gdf.iterrows():
    ax.text(
        row.geometry.x + x_offset,
        row.geometry.y,
        row["Name"],
        fontsize=8,
        ha="left",
        va="center"
    )

# Basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

# --- Compute true bounds of the filtered substations ---
xmin, ymin, xmax, ymax = gdf.total_bounds

# --- Apply a small zoom-out buffer (in metres, EPSG:3857) ---
x_buffer = 3500        # slight horizontal zoom-out
y_buffer_top = 3000    # a bit more room above Umina
y_buffer_bottom = 800  # minimal room below

extra_bottom = 1500

ax.set_xlim(xmin - x_buffer, xmax + x_buffer)
ax.set_ylim(ymin - (y_buffer_bottom + extra_bottom), ymax + y_buffer_top)

# -----------------------------
# Tick conversion (AFTER padding)
# -----------------------------
xticks_3857 = ax.get_xticks()
yticks_3857 = ax.get_yticks()

to_lonlat = pyproj.Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

lon_ticks, _ = to_lonlat.transform(xticks_3857, np.zeros_like(xticks_3857))
_, lat_ticks = to_lonlat.transform(np.zeros_like(yticks_3857), yticks_3857)

ax.xaxis.set_major_locator(FixedLocator(xticks_3857))
ax.yaxis.set_major_locator(FixedLocator(yticks_3857))

ax.set_xticklabels([f"{lon:.3f}" for lon in lon_ticks])
ax.set_yticklabels([f"{lat:.3f}" for lat in lat_ticks])

plt.setp(ax.get_xticklabels(), rotation=45, ha="right")


ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.title("Substations with Residential Fraction ≥ 0.75")

plt.tight_layout()
plt.close()


## Mapping different residential fractions
- greater than 0.75 as a circle, 0.65 - 0.75 as a square and 0.55 - 0.65 as a triangle

In [ ]:
def plot_substations(info):
    import geopandas as gpd
    import matplotlib.pyplot as plt
    import contextily as ctx
    import pyproj
    import numpy as np
    from matplotlib.ticker import FixedLocator

    # -----------------------------
    # Split into three marker groups
    # -----------------------------
    g_circle = info[info["Residential"] >= 0.75].copy()
    g_square = info[(info["Residential"] >= 0.65) & (info["Residential"] < 0.75)].copy()
    g_triangle = info[(info["Residential"] >= 0.55) & (info["Residential"] < 0.65)].copy()

    # -----------------------------
    # Convert to GeoDataFrames
    # -----------------------------
    def make_gdf(df):
        return gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df.longitude, df.latitude),
            crs="EPSG:4326"
        ).to_crs("EPSG:3857")

    g_circle = make_gdf(g_circle)
    g_square = make_gdf(g_square)
    g_triangle = make_gdf(g_triangle)

    # Combine for bounds
    all_gdf = gpd.GeoDataFrame(
        pd.concat([g_circle, g_square, g_triangle], ignore_index=True),
        geometry="geometry",
        crs="EPSG:3857"
    )

    # -----------------------------
    # Plot (auto-zoom)
    # -----------------------------
    fig, ax = plt.subplots(figsize=(8, 10))

    # Marker groups
    g_circle.plot(ax=ax, color="red",   markersize=40, marker="o", label="≥ 0.75")
    g_square.plot(ax=ax, color="blue",  markersize=40, marker="s", label="0.65–0.75")
    g_triangle.plot(ax=ax, color="green", markersize=40, marker="^", label="0.55–0.65")

    # -----------------------------
    # Labels (600 m to the right)
    # -----------------------------
    x_offset = 600

    for _, row in all_gdf.iterrows():
        ax.text(
            row.geometry.x + x_offset,
            row.geometry.y,
            row["Name"],
            fontsize=8,
            ha="left",
            va="center"
        )

    # -----------------------------
    # Basemap
    # -----------------------------
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

    # -----------------------------
    # Bounds + padding
    # -----------------------------
    xmin, ymin, xmax, ymax = all_gdf.total_bounds

    x_buffer = 3500
    y_buffer_top = 3000
    y_buffer_bottom = 800
    extra_bottom = 4000

    ax.set_xlim(xmin - x_buffer, xmax + x_buffer)
    ax.set_ylim(ymin - (y_buffer_bottom + extra_bottom), ymax + y_buffer_top)

    # -----------------------------
    # Tick conversion (AFTER padding)
    # -----------------------------
    xticks_3857 = ax.get_xticks()
    yticks_3857 = ax.get_yticks()

    to_lonlat = pyproj.Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

    lon_ticks, _ = to_lonlat.transform(xticks_3857, np.zeros_like(xticks_3857))
    _, lat_ticks = to_lonlat.transform(np.zeros_like(yticks_3857), yticks_3857)

    ax.xaxis.set_major_locator(FixedLocator(xticks_3857))
    ax.yaxis.set_major_locator(FixedLocator(yticks_3857))

    ax.set_xticklabels([f"{lon:.3f}" for lon in lon_ticks])
    ax.set_yticklabels([f"{lat:.3f}" for lat in lat_ticks])

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    # -----------------------------
    # Labels & title
    # -----------------------------
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Substations by Residential Fraction")
    ax.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
plot_substations(info)


### Rewriting above to have substation representatives
- using SA2
- median residential fractions per defined region rather than suburb

In [ ]:
region_map = {
    # Newcastle region
    "Adamstown": "Newcastle",
    "Mayfield West": "Newcastle",
    "Maryland": "Newcastle",
    "Argenton": "Newcastle",
    "Nelson Bay": "Newcastle",
    "Muswellbrook": "Newcastle",

    # Lake Macquarie
    "Morisset": "Lake Macquarie",
    "Charmhaven": "Lake Macquarie",
    "Cardiff": "Lake Macquarie",

    # Central Coast
    "Umina": "Central Coast",
    "Woy Woy": "Central Coast",
    "Lisarow": "Central Coast",

    # Northern Sydney
    "Narrabeen": "Northern Sydney",
    "Mosman": "Northern Sydney",
    "Meadowbank": "Northern Sydney",
    "Macquarie Park": "Northern Sydney",

    # Eastern Sydney
    "Mascot": "Eastern Sydney",
    "Maroubra": "Eastern Sydney",
    "Matraville": "Eastern Sydney",

    # Inner West
    "Marrickville": "Inner West",

    # Western Sydney
    "Lidcombe": "Western Sydney",
    "Leightonfield": "Western Sydney",
    "Milperra": "Western Sydney",

    # Add the rest of your suburbs here
}


In [ ]:
def plot_region_median(info, region_map):
    import geopandas as gpd
    import matplotlib.pyplot as plt
    import contextily as ctx
    import pyproj
    import numpy as np
    import pandas as pd
    from matplotlib.ticker import FixedLocator

    # -----------------------------
    # Assign each suburb to a region
    # -----------------------------
    info = info.copy()
    info["Region"] = info["Name"].map(region_map).fillna("Other")

    # -----------------------------
    # Aggregate by region
    # -----------------------------
    rep = (
        info.groupby("Region")
            .agg({
                "Residential": "median",
                "latitude": "median",
                "longitude": "median"
            })
            .reset_index()
            .rename(columns={"Residential": "MedianResidential"})
    )

    # -----------------------------
    # Convert to GeoDataFrame
    # -----------------------------
    gdf = gpd.GeoDataFrame(
        rep,
        geometry=gpd.points_from_xy(rep.longitude, rep.latitude),
        crs="EPSG:4326"
    ).to_crs("EPSG:3857")

    # -----------------------------
    # Plot
    # -----------------------------
    fig, ax = plt.subplots(figsize=(8, 10))

    gdf.plot(
        ax=ax,
        column="MedianResidential",
        cmap="viridis",
        markersize=200,
        legend=True,
        legend_kwds={"label": "Median Residential Fraction"},
    )

    # -----------------------------
    # Labels
    # -----------------------------
    for _, row in gdf.iterrows():
        ax.text(
            row.geometry.x + 800,
            row.geometry.y,
            row["Region"],
            fontsize=9,
            ha="left",
            va="center"
        )

    # -----------------------------
    # Basemap
    # -----------------------------
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

    # -----------------------------
    # Bounds + padding
    # -----------------------------
    xmin, ymin, xmax, ymax = gdf.total_bounds

    x_buffer = 5000
    y_buffer_top = 5000
    y_buffer_bottom = 2000

    ax.set_xlim(xmin - x_buffer, xmax + x_buffer)
    ax.set_ylim(ymin - y_buffer_bottom, ymax + y_buffer_top)

    # -----------------------------
    # Tick conversion
    # -----------------------------
    xticks_3857 = ax.get_xticks()
    yticks_3857 = ax.get_yticks()

    to_lonlat = pyproj.Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

    lon_ticks, _ = to_lonlat.transform(xticks_3857, np.zeros_like(xticks_3857))
    _, lat_ticks = to_lonlat.transform(np.zeros_like(yticks_3857), yticks_3857)

    ax.xaxis.set_major_locator(FixedLocator(xticks_3857))
    ax.yaxis.set_major_locator(FixedLocator(yticks_3857))

    ax.set_xticklabels([f"{lon:.3f}" for lon in lon_ticks])
    ax.set_yticklabels([f"{lat:.3f}" for lat in lat_ticks])

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    # -----------------------------
    # Labels & title
    # -----------------------------
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Median Residential Fraction by Region")

    plt.tight_layout()
    plt.show()


In [ ]:
plot_region_median(info, region_map)

## Zooming into the Sydney region

In [ ]:
sydney_micro_map = {
    # -------------------------
    # Northern Beaches North
    # -------------------------
    "Narrabeen": "Northern Beaches North",
    "Dee Why West": "Northern Beaches North",
    "Harbord": "Northern Beaches North",
    "Beacon Hill": "Northern Beaches North",

    # -------------------------
    # Northern Beaches South
    # -------------------------
    "Balgowlah North": "Northern Beaches South",
    "Brookvale": "Northern Beaches South",

    # -------------------------
    # Lower North Shore
    # -------------------------
    "Mosman": "Lower North Shore",
    "Castle Cove": "Lower North Shore",
    "Chatswood": "Lower North Shore",
    "Gore Hill": "Lower North Shore",
    "Hunters Hill": "Lower North Shore",
    "Pymble": "Lower North Shore",

    # -------------------------
    # Macquarie / Ryde Corridor
    # -------------------------
    "Macquarie Park": "Macquarie Corridor",
    "Meadowbank": "Macquarie Corridor",
    "Epping": "Macquarie Corridor",
    "Top Ryde": "Macquarie Corridor",

    # -------------------------
    # Inner West North
    # -------------------------
    "Five Dock": "Inner West North",
    "Burwood": "Inner West North",
    "Concord": "Inner West North",
    "Enfield": "Inner West North",

    # -------------------------
    # Inner West South
    # -------------------------
    "Dulwich Hill": "Inner West South",
    "Marrickville": "Inner West South",
    "St. Peters": "Inner West South",
    "Campsie": "Inner West South",

    # -------------------------
    # Eastern Suburbs North
    # -------------------------
    "Green Square": "Eastern Suburbs North",
    "Zetland": "Eastern Suburbs North",
    "Mascot": "Eastern Suburbs North",

    # -------------------------
    # Eastern Suburbs South
    # -------------------------
    "Maroubra": "Eastern Suburbs South",
    "Matraville": "Eastern Suburbs South",
    "Botany": "Eastern Suburbs South",

    # -------------------------
    # Inner South North (St George North)
    # -------------------------
    "Arncliffe": "Inner South North",
    "Rockdale": "Inner South North",
    "Kogarah": "Inner South North",

    # -------------------------
    # Inner South South (St George South)
    # -------------------------
    "Mortdale": "Inner South South",
    "Blakehurst": "Inner South South",

    # -------------------------
    # Sutherland Shire
    # -------------------------
    "Caringbah": "Sutherland Shire",
    "Cronulla": "Sutherland Shire",
    "Kirrawee": "Sutherland Shire",

    # -------------------------
    # Inner West–West (Strathfield / Homebush belt)
    # -------------------------
    "Homebush Bay": "Inner West–West",
    "Flemington": "Inner West–West",
    "Lidcombe": "Inner West–West",
    "Leightonfield": "Inner West–West",

    # -------------------------
    # Bankstown Core
    # -------------------------
    "Bankstown": "Bankstown Core",
    "Bass Hill": "Bankstown Core",
    "Greenacre Park": "Bankstown Core",
    "Potts Hill": "Bankstown Core",

    # -------------------------
    # Canterbury–Revesby Corridor
    # -------------------------
    "Punchbowl": "Canterbury–Revesby Corridor",
    "Riverwood": "Canterbury–Revesby Corridor",
    "Revesby": "Canterbury–Revesby Corridor",
    "Sefton": "Canterbury–Revesby Corridor",
    "Milperra": "Canterbury–Revesby Corridor",

    # -------------------------
    # Western Sydney (remaining)
    # -------------------------
    "Auburn": "Western Sydney",

    # -------------------------
    # Northern Fringe
    # -------------------------
    "Pennant Hills": "Northern Fringe",
    "Hornsby": "Northern Fringe",
    "Berowra": "Northern Fringe",
}


In [ ]:
def plot_sydney_subregions_detailed(info, sydney_region_map):
    import geopandas as gpd
    import matplotlib.pyplot as plt
    import contextily as ctx
    import pyproj
    import numpy as np
    import pandas as pd
    from matplotlib.ticker import FixedLocator
    from mpl_toolkits.axes_grid1 import make_axes_locatable

    # -----------------------------
    # Assign micro‑regions
    # -----------------------------
    info = info.copy()
    info["SydneyRegion"] = info["Name"].map(sydney_region_map)
    syd = info.dropna(subset=["SydneyRegion"])

    # -----------------------------
    # Aggregate by micro‑region
    # -----------------------------
    rep = (
        syd.groupby("SydneyRegion")
            .agg({
                "Residential": "median",
                "latitude": "median",
                "longitude": "median"
            })
            .reset_index()
            .rename(columns={"Residential": "MedianResidential"})
    )

    # -----------------------------
    # GeoDataFrame
    # -----------------------------
    gdf = gpd.GeoDataFrame(
        rep,
        geometry=gpd.points_from_xy(rep.longitude, rep.latitude),
        crs="EPSG:4326"
    ).to_crs("EPSG:3857")

    # -----------------------------
    # Bounds FIRST (prevents errors)
    # -----------------------------
    xmin, ymin, xmax, ymax = gdf.total_bounds
    x_range = xmax - xmin
    y_range = ymax - ymin

    # -----------------------------
    # Auto‑wrap long labels
    # -----------------------------
    def wrap_label(name, max_len=13):
        if len(name) <= max_len:
            return name
        parts = name.split()
        mid = len(parts) // 2
        return " ".join(parts[:mid]) + "\n" + " ".join(parts[mid:])

    # -----------------------------
    # Plot
    # -----------------------------
    fig, ax = plt.subplots(figsize=(8, 10))

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3%", pad=1.2)  # legend moved right

    sc = gdf.plot(
        ax=ax,
        column="MedianResidential",
        cmap="YlOrRd",
        markersize=260,
        legend=False
    )

    cb = plt.colorbar(sc.collections[0], cax=cax)
    cb.set_label("Median Residential Fraction")

    # -----------------------------
    # Label placement (scaled offsets)
    # -----------------------------
    x_off = 0.015 * x_range
    y_off = 0.015 * y_range

    for _, row in gdf.iterrows():
        label = wrap_label(row["SydneyRegion"])
        ax.text(
            row.geometry.x + x_off,
            row.geometry.y + y_off,
            label,
            fontsize=9,
            ha="left",
            va="center"
        )

    # -----------------------------
    # Basemap
    # -----------------------------
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

    # -----------------------------
    # Dynamic extent (no clipping)
    # -----------------------------
    ax.set_xlim(xmin - 0.05 * x_range, xmax + 0.15 * x_range)
    ax.set_ylim(ymin - 0.10 * y_range, ymax + 0.10 * y_range)

    # -----------------------------
    # Tick conversion
    # -----------------------------
    xticks_3857 = ax.get_xticks()
    yticks_3857 = ax.get_yticks()

    to_lonlat = pyproj.Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
    lon_ticks, _ = to_lonlat.transform(xticks_3857, np.zeros_like(xticks_3857))
    _, lat_ticks = to_lonlat.transform(np.zeros_like(yticks_3857), yticks_3857)

    ax.xaxis.set_major_locator(FixedLocator(xticks_3857))
    ax.yaxis.set_major_locator(FixedLocator(yticks_3857))

    ax.set_xticklabels([f"{lon:.3f}" for lon in lon_ticks])
    ax.set_yticklabels([f"{lat:.3f}" for lat in lat_ticks])

    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Median Residential Fraction — Detailed Sydney Sub‑Regions")

    plt.tight_layout()
    plt.show()


In [ ]:
plot_sydney_subregions_detailed(info, sydney_micro_map)

### Adding in geographic spread dot sizes
- using polygons
- can't get to completely work

In [ ]:
region_codes = {
    "Northern Fringe": "NF",
    "Macquarie Corridor": "MC",
    "Lower North Shore": "LNS",
    "Inner West North": "IWN",
    "Inner West South": "IWS",
    "Eastern Suburbs North": "ESN",
    "Eastern Suburbs South": "ESS",
    "Sutherland Shire": "SS",
    "Bankstown Core": "BC",
    "Canterbury-Revesby Corridor": "CRC",
    "Inner South South": "INSS",
    # add the rest of your micro‑regions here
}


In [ ]:
import geopandas as gpd

suburbs = gpd.read_file(
    "/g/data/ng72/pv3484/substation_data/natural_earth_coastline/ssc_2016/SSC_2016_AUST.shp"
)

suburbs.head()


In [ ]:
suburbs_nsw = suburbs[suburbs["STE_NAME16"] == "New South Wales"]

suburbs_sydney = suburbs_nsw[
    (suburbs_nsw.geometry.bounds.minx > 150.0) &
    (suburbs_nsw.geometry.bounds.maxx < 151.5) &
    (suburbs_nsw.geometry.bounds.miny > -34.1) &
    (suburbs_nsw.geometry.bounds.maxy < -33.5)
]



In [ ]:
def plot_sydney_microregion_polygons(info, suburbs, sydney_micro_map, region_codes):
    import geopandas as gpd
    import matplotlib.pyplot as plt
    import contextily as ctx
    import pyproj
    import numpy as np
    import pandas as pd
    import matplotlib.patheffects as pe
    from matplotlib.ticker import FixedLocator
    from mpl_toolkits.axes_grid1 import make_axes_locatable

    # -----------------------------
    # 1. Assign micro‑regions to substations
    # -----------------------------
    info = info.copy()
    info["SydneyRegion"] = info["Name"].map(sydney_micro_map)
    syd = info.dropna(subset=["SydneyRegion"])

    rep = (
        syd.groupby("SydneyRegion")["Residential"]
            .median()
            .rename("MedianResidential")
            .reset_index()
    )

    # -----------------------------
    # 2. Filter SSC polygons to NSW + Sydney metro
    # -----------------------------
    suburbs = suburbs.copy()
    suburbs = suburbs[suburbs["STE_NAME16"] == "New South Wales"]
    suburbs = suburbs.cx[150.5:151.4, -34.1:-33.5]

    # -----------------------------
    # 3. Attach micro‑regions to suburb polygons
    # -----------------------------
    suburbs["SydneyRegion"] = suburbs["SSC_NAME16"].map(sydney_micro_map)
    syd_poly = suburbs.dropna(subset=["SydneyRegion"])

    # -----------------------------
    # 4. Merge residential medians
    # -----------------------------
    syd_poly = syd_poly.merge(rep, on="SydneyRegion", how="left")

    # Remove duplicate columns created by merge
    syd_poly = syd_poly.loc[:, ~syd_poly.columns.duplicated()]

    # -----------------------------
    # 5. Dissolve polygons into micro‑regions
    # -----------------------------
    regions = syd_poly[["SydneyRegion", "MedianResidential", "geometry"]].dissolve(
        by="SydneyRegion",
        aggfunc={"MedianResidential": "mean"}
    )

    # Ensure numeric dtype
    regions["MedianResidential"] = pd.to_numeric(regions["MedianResidential"], errors="coerce")

    # Replace missing values with global median
    regions["MedianResidential"] = regions["MedianResidential"].fillna(
        regions["MedianResidential"].median()
    )

    # Keep raw geometry (no simplify)
    regions["geometry"] = regions.buffer(0)

    # Project for plotting
    regions = regions.to_crs("EPSG:3857")
    syd_poly = syd_poly.to_crs("EPSG:3857")

    xmin, ymin, xmax, ymax = regions.total_bounds
    x_range = xmax - xmin
    y_range = ymax - ymin

    # -----------------------------
    # 6. Plot setup
    # -----------------------------
    fig, ax = plt.subplots(figsize=(9, 10))
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3%", pad=1.2)

    # Subtle suburb outlines
    syd_poly.plot(
        ax=ax,
        facecolor="none",
        edgecolor="grey",
        linewidth=0.15,
        alpha=0.35
    )

    # Region polygons
    regions.plot(
        ax=ax,
        column="MedianResidential",
        cmap="YlOrRd",
        linewidth=1.2,
        edgecolor="white",
        alpha=0.85,
        legend=False
    )

    # -----------------------------
    # 7. Colourbar
    # -----------------------------
    sm = plt.cm.ScalarMappable(
        cmap="YlOrRd",
        norm=plt.Normalize(
            vmin=regions["MedianResidential"].min(),
            vmax=regions["MedianResidential"].max()
        )
    )
    sm._A = []
    cb = plt.colorbar(sm, cax=cax)
    cb.set_label("Median Residential Fraction")

    # -----------------------------
    # 8. Shorthand labels in *every* polygon piece
    # -----------------------------
    regions_exploded = regions.explode(index_parts=False)
    rep_points = regions_exploded.representative_point()

    for idx, row in regions_exploded.iterrows():
        region_name = idx
        code = region_codes.get(region_name, "")
        point = rep_points.loc[idx]

        ax.text(
            point.x,
            point.y,
            code,
            fontsize=11,
            fontweight="bold",
            ha="center",
            va="center",
            color="black",
            path_effects=[pe.withStroke(linewidth=2.5, foreground="white")]
        )

    # -----------------------------
    # 9. Region key beside colourbar
    # -----------------------------
    key_text = "\n".join([f"{code} — {name}" for name, code in region_codes.items()])

    fig.text(
        0.97,
        0.50,
        key_text,
        va="center",
        ha="left",
        fontsize=9.5,
        fontweight="medium",
        linespacing=1.4
    )

    # -----------------------------
    # 10. Basemap
    # -----------------------------
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronNoLabels)

    # -----------------------------
    # 11. Tightened extents
    # -----------------------------
    ax.set_xlim(xmin - 0.02 * x_range, xmax + 0.05 * x_range)
    ax.set_ylim(ymin - 0.02 * y_range, ymax + 0.02 * y_range)

    # -----------------------------
    # 12. Longitude/latitude ticks
    # -----------------------------
    xticks_3857 = ax.get_xticks()
    yticks_3857 = ax.get_yticks()

    to_lonlat = pyproj.Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
    lon_ticks, _ = to_lonlat.transform(xticks_3857, np.zeros_like(xticks_3857))
    _, lat_ticks = to_lonlat.transform(np.zeros_like(yticks_3857), yticks_3857)

    ax.xaxis.set_major_locator(FixedLocator(xticks_3857))
    ax.yaxis.set_major_locator(FixedLocator(yticks_3857))

    ax.set_xticklabels([f"{lon:.3f}" for lon in lon_ticks])
    ax.set_yticklabels([f"{lat:.3f}" for lat in lat_ticks])
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Median Residential Fraction — Sydney Micro‑Region Polygons", pad=15)

    plt.tight_layout()
    plt.show()


In [ ]:
info.columns


# Substations and Weather Stations
- for seminar
- mapping all substations and weather stations without name locations
- scale bar and north arrow

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib_scalebar.scalebar import ScaleBar
import numpy as np

# 1. Convert demand substations to GeoDataFrame
gdf_substations = gpd.GeoDataFrame(
    info[['latitude', 'longitude']].drop_duplicates(),
    geometry=gpd.points_from_xy(
        info[['latitude', 'longitude']].drop_duplicates().longitude,
        info[['latitude', 'longitude']].drop_duplicates().latitude
    ),
    crs="EPSG:4326"
)

# 2. Convert weather stations to GeoDataFrame
weather_locs = weather[['lat', 'lon']].drop_duplicates()
gdf_weather = gpd.GeoDataFrame(
    weather_locs,
    geometry=gpd.points_from_xy(weather_locs.lon, weather_locs.lat),
    crs="EPSG:4326"
)

# 3. Plot both layers
fig, ax = plt.subplots(figsize=(10, 12))
gdf_substations.plot(ax=ax, color="red", marker="o", markersize=20, alpha=0.8, label="Demand Substations")
gdf_weather.plot(ax=ax, color="blue", marker="s", markersize=30, alpha=0.8, label="Weather Stations")

# 4. Add basemap
ctx.add_basemap(ax, crs=gdf_substations.crs, source=ctx.providers.CartoDB.Positron)

# 5. Scale bar — axes are in degrees but scalebar displays in km
lat_sydney = -33.9
metres_per_degree = 111320 * np.cos(np.radians(lat_sydney))  # ~92,000 m per degree

scalebar = ScaleBar(
    dx=metres_per_degree,
    units="m",
    dimension="si-length",
    length_fraction=0.2,
    location="lower right",
    box_alpha=0.7,
    font_properties={"size": 9}
)
ax.add_artist(scalebar)

# 6. North arrow
x, y, arrow_length = 0.95, 0.95, 0.06
ax.annotate(
    'N',
    xy=(x, y),
    xytext=(x, y - arrow_length),
    xycoords='axes fraction',
    textcoords='axes fraction',
    ha='center',
    va='center',
    fontsize=14,
    fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='black', lw=2),
)

# 7. Legend, titles, labels
ax.legend(loc="upper left", fontsize=10)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.title("Substation and Weather Station Locations")
plt.tight_layout()
# plt.savefig("/home/565/pv3484/aus_substation_electricity/data/figures/substation_weather_map.png", dpi=300, bbox_inches="tight")
plt.show()

## Same map but with punchbowl and dulwich hill (DU_HI) highlighted by different symbols

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
import os

output_dir = "/home/565/pv3484/aus_substation_electricity/figures/"
os.makedirs(output_dir, exist_ok=True)

# --- Deduplicate locations ---
info_locs = info[['Name', 'latitude', 'longitude']].drop_duplicates()
weather_locs = weather[['lat', 'lon']].drop_duplicates()

# --- Split out Dulwich Hill and Punchbowl ---
special = info_locs[info_locs['Name'].isin(['Dulwich Hill', 'Punchbowl'])]
regular = info_locs[~info_locs['Name'].isin(['Dulwich Hill', 'Punchbowl'])]

# --- Find closest weather station for each special substation ---
def nearest_weather(substation_row, weather_locs):
    dists = ((weather_locs.lat - substation_row.latitude)**2 +
             (weather_locs.lon - substation_row.longitude)**2)
    return weather_locs.loc[dists.idxmin()]

du_hi_row    = special[special['Name'] == 'Dulwich Hill'].iloc[0]
punchbowl_row = special[special['Name'] == 'Punchbowl'].iloc[0]

du_hi_wx     = nearest_weather(du_hi_row, weather_locs)
punchbowl_wx = nearest_weather(punchbowl_row, weather_locs)

# Remaining weather stations
special_wx = pd.DataFrame([du_hi_wx, punchbowl_wx])
regular_wx = weather_locs.merge(special_wx, on=['lat', 'lon'], how='left', indicator=True)
regular_wx = regular_wx[regular_wx['_merge'] == 'left_only'].drop(columns='_merge')

# --- Build GeoDataFrames ---
def make_gdf(df, lon_col, lat_col):
    return gpd.GeoDataFrame(
        df, geometry=gpd.points_from_xy(df[lon_col], df[lat_col]), crs="EPSG:4326"
    )

gdf_regular      = make_gdf(regular, 'longitude', 'latitude')
gdf_du_hi        = make_gdf(special[special['Name'] == 'Dulwich Hill'], 'longitude', 'latitude')
gdf_punchbowl    = make_gdf(special[special['Name'] == 'Punchbowl'], 'longitude', 'latitude')
gdf_regular_wx   = make_gdf(regular_wx, 'lon', 'lat')
gdf_du_hi_wx     = make_gdf(pd.DataFrame([du_hi_wx]), 'lon', 'lat')
gdf_punchbowl_wx = make_gdf(pd.DataFrame([punchbowl_wx]), 'lon', 'lat')

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 12))

gdf_regular.plot(ax=ax, color="red", marker="o", markersize=20, alpha=0.8, label="Demand Substations")
gdf_regular_wx.plot(ax=ax, color="blue", marker="s", markersize=30, alpha=0.8, label="Weather Stations")
gdf_du_hi.plot(ax=ax, color="teal", marker="*", markersize=250, alpha=0.9, label="Dulwich Hill Substation")
gdf_punchbowl.plot(ax=ax, color="darkgreen", marker="*", markersize=250, alpha=0.9, label="Punchbowl Substation")
gdf_du_hi_wx.plot(ax=ax, color="teal", marker="^", markersize=80, alpha=0.9, label="Dulwich Hill Nearest Weather Station")
gdf_punchbowl_wx.plot(ax=ax, color="darkgreen", marker="^", markersize=80, alpha=0.9, label="Punchbowl Nearest Weather Station")

ctx.add_basemap(ax, crs="EPSG:4326", source=ctx.providers.CartoDB.Positron)
ax.legend(loc="upper left", fontsize=9)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.title("Substation and Weather Station Locations")
plt.tight_layout()
# plt.savefig("/home/565/pv3484/aus_substation_electricity/data/figures/substation_weather_map_seminar.png", dpi=300, bbox_inches="tight")
plt.show()

## Zoom around mosman and punchbowl
- adding scale bar and north arrow

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import contextily as ctx
import pandas as pd
import os
import numpy as np
from matplotlib_scalebar.scalebar import ScaleBar

output_dir = "/home/565/pv3484/aus_substation_electricity/data/figures/"
os.makedirs(output_dir, exist_ok=True)

# --- Deduplicate locations ---
info_locs = info[['Name', 'latitude', 'longitude']].drop_duplicates()
weather_locs = weather[['lat', 'lon']].drop_duplicates()

# --- Split out Dulwich Hill and Punchbowl ---
special = info_locs[info_locs['Name'].isin(['Dulwich Hill', 'Punchbowl'])]
regular = info_locs[~info_locs['Name'].isin(['Dulwich Hill', 'Punchbowl'])]

du_hi_row     = special[special['Name'] == 'Dulwich Hill'].iloc[0]
punchbowl_row = special[special['Name'] == 'Punchbowl'].iloc[0]

# --- Find closest weather station for each special substation ---
def nearest_weather(substation_row, weather_locs):
    dists = ((weather_locs.lat - substation_row.latitude)**2 +
             (weather_locs.lon - substation_row.longitude)**2)
    return weather_locs.loc[dists.idxmin()]

du_hi_wx     = nearest_weather(du_hi_row, weather_locs)
punchbowl_wx = nearest_weather(punchbowl_row, weather_locs)

# --- Split out special weather stations from regular ---
special_wx = pd.DataFrame([du_hi_wx, punchbowl_wx])
regular_wx = weather_locs.merge(special_wx, on=['lat', 'lon'], how='left', indicator=True)
regular_wx = regular_wx[regular_wx['_merge'] == 'left_only'].drop(columns='_merge')

# --- Helper to build GeoDataFrames ---
def make_gdf(df, lon_col, lat_col):
    return gpd.GeoDataFrame(
        df, geometry=gpd.points_from_xy(df[lon_col], df[lat_col]), crs="EPSG:4326"
    )

gdf_regular      = make_gdf(regular, 'longitude', 'latitude')
gdf_du_hi        = make_gdf(special[special['Name'] == 'Dulwich Hill'], 'longitude', 'latitude')
gdf_punchbowl    = make_gdf(special[special['Name'] == 'Punchbowl'], 'longitude', 'latitude')
gdf_regular_wx   = make_gdf(regular_wx, 'lon', 'lat')
gdf_du_hi_wx     = make_gdf(pd.DataFrame([du_hi_wx]), 'lon', 'lat')
gdf_punchbowl_wx = make_gdf(pd.DataFrame([punchbowl_wx]), 'lon', 'lat')

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 12))

gdf_regular.plot(ax=ax, color="red", marker="o", markersize=30, alpha=0.8, label="Demand Substations")
gdf_regular_wx.plot(ax=ax, color="blue", marker="s", markersize=30, alpha=0.8, label="Weather Stations")
gdf_du_hi.plot(ax=ax, color="darkcyan", marker="*", markersize=300, alpha=0.9, label="Dulwich Hill Substation")
gdf_punchbowl.plot(ax=ax, color="green", marker="*", markersize=300, alpha=0.9, label="Punchbowl Substation")
gdf_du_hi_wx.plot(ax=ax, color="purple", marker="^", markersize=300, alpha=0.9, label="Shared Nearest Weather Station")
gdf_punchbowl_wx.plot(ax=ax, color="purple", marker="^", markersize=300, alpha=0.9)  # same station, no second label

# --- Zoom to Dulwich Hill and Punchbowl ---
lons = [du_hi_row.longitude, punchbowl_row.longitude]
lats = [du_hi_row.latitude, punchbowl_row.latitude]
padding = 0.15

ax.set_xlim(min(lons) - padding, max(lons) + padding)
ax.set_ylim(min(lats) - padding, max(lats) + padding)

# --- Basemap ---
ctx.add_basemap(ax, crs="EPSG:4326", source=ctx.providers.CartoDB.Positron)

# --- Scale bar ---
lat_sydney = -33.9
metres_per_degree = 111320 * np.cos(np.radians(lat_sydney))

scalebar = ScaleBar(
    dx=metres_per_degree,
    units="m",
    dimension="si-length",
    length_fraction=0.2,
    location="lower right",
    box_alpha=0.7,
    font_properties={"size": 9}
)
ax.add_artist(scalebar)

# --- North arrow ---
x, y, arrow_length = 0.95, 0.95, 0.06
ax.annotate(
    'N',
    xy=(x, y),
    xytext=(x, y - arrow_length),
    xycoords='axes fraction',
    textcoords='axes fraction',
    ha='center',
    va='center',
    fontsize=14,
    fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='black', lw=2),
)

# --- Legend, labels ---
ax.legend(loc="upper left", fontsize=9)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.title("Substation and Weather Station Locations (Zoomed: Dulwich Hill & Punchbowl)")
plt.tight_layout()
plt.savefig(f"{output_dir}substation_weather_map_zoomed.png", dpi=300, bbox_inches="tight")
plt.show()